
# Introduction

# Albotherm AXF — Data Merge
### Pipeline Step 2 of 4: Clean → **Merge** → EDA → Model

---

## Introduction

Albotherm manufactures AXF microcapsules — a temperature responsive delivery
system in which a hydrogel core is encapsulated within a UV cured polymer shell.
Each production run (AXF) combines three independently formulated materials:
a **core**, a **UV shell**, and an **outer phase**, each produced in discrete batches before assembly.

This notebook merges the cleaned datasets (in step 1) into a single analysis ready structure.
The goal is to link every AXF production run to its durability outcome, the
specific batches used, and the formulation identities of those batches  without introducing data redundancy.

### Why merging this data is complex

The data spans three levels of depth:

| Level | Example | Relationship |
|---|---|---|
| Formulation recipe | `RAD-C-0001` | One recipe → many batches |
| Batch | `BNC00001` | One batch → many AXF runs |
| AXF run | `AXF0020` | One run → one durability result |

A naive flat merge across all files would cause **row explosion** and **column redundancy** — for example, merging 88 UV ingredient columns into every AXF row
when only the formulation ID is needed. Instead, we follow the Tidy Data
principle (Wickham, 2014) and separate *lookup* tables from the *analysis table*.

### Datasets used in this notebook are all from cleaned datasets from step 1

| Variable | File | Rows | Description |
|---|---|---|---|
| `mix` | `mix_cleaned.csv` | 628 | AXF production runs — the spine |
| `result` | `result_cleaned.csv` | 679 | Durability & QC outcomes |
| `core_batch` | `core_batch_cleaned.csv` | 117 | QC per core batch |
| `uv_batch` | `uv_batch_cleaned.csv` | 177 | QC per UV batch |
| `outer_batch` | `outer_batch_cleaned.csv` | 255 | QC per outer batch |
| `core` | `core_cleaned.csv` | 54 | Core ingredient recipes *(lookup)* |
| `uv` | `uv_cleaned.csv` | 323 | UV ingredient recipes *(lookup)* |
| `outer` | `outer_cleaned.csv` | 7 | Outer ingredient recipes *(lookup)* |

---



> Wickham, H. (2014). Tidy Data. *Journal of Statistical Software*, 59(10).
> https://doi.org/10.18637/jss.v059.i10






## 1. Loading All cleaned files




In [ ]:
import pandas as pd

# We Load the cleaned datasets

uv          = pd.read_csv('../data/processed/uv_cleaned.csv')
core        = pd.read_csv('../data/processed/core_cleaned.csv')
outer       = pd.read_csv('../data/processed/outer_cleaned.csv')

# starting materials dataset also has 3 different types of batch, corresponding to 3 formulas.
core_batch  = pd.read_csv('../data/processed/core_batch_cleaned.csv')
uv_batch    = pd.read_csv('../data/processed/uv_batch_cleaned.csv')
outer_batch = pd.read_csv('../data/processed/outer_batch_cleaned.csv')

# axf dataset is the primary production data, including the combination of main materials (UV, core, outer) along with other materials and surrounding conditions.
mix         = pd.read_csv('../data/processed/mix_cleaned.csv')

# durability dataset show the quality check result of each axf combination.
result      = pd.read_csv('../data/processed/result_cleaned.csv')

print("All files loaded\n")
for name, df in [("uv", uv), ("core", core), ("outer", outer),
                 ("core_batch", core_batch), ("uv_batch", uv_batch),
                 ("outer_batch", outer_batch), ("mix", mix), ("result", result)]:
    print(f"  {name:15s}: {df.shape}")


Mounted at /content/drive
All files loaded

  uv             : (323, 89)
  core           : (54, 23)
  outer          : (7, 8)
  core_batch     : (117, 12)
  uv_batch       : (177, 9)
  outer_batch    : (255, 9)
  mix            : (628, 33)
  result         : (679, 15)


## 2. Identifying Key Columns

Before merging, we establish which column in each table serves as the join key — the shared identifier that links tables together. These are not always obvious from column names alone, particularly where naming is inconsistent across files.

We inspect the key column from each table and print example values to confirm they look as expected.

mix and result now share a pre-built `batch_key` column created in `albotherm_01_clean.ipynb` with
consistent zero-padding and extra texts removed. The key inspection below confirms
this is present and clean — no further whitespace cleaning is needed for these two tables.

In [2]:
key_map = {
    'mix'         : 'batch_key',
    'result'      : 'batch_key',
    'core_batch'  : 'Batch Number ',
    'uv_batch'    : 'Batch Number ',
    'outer_batch' : 'Batch Number ',
    'core'        : 'Sample',
    'uv'          : 'Sample',
    'outer'       : 'Sample',
}

print(f"{'Table':<15} {'Key column':<25} {'Example values'}")
print("-" * 75)
for table_name, key_col in key_map.items():
    df = eval(table_name)
    examples = df[key_col].dropna().unique()[:3].tolist()
    print(f"{table_name:<15} {repr(key_col):<25} {examples}")


Table           Key column                Example values
---------------------------------------------------------------------------
mix             'batch_key'               ['AXF0020', 'AXF0021', 'AXF0022']
result          'batch_key'               ['AXF0340', 'AXF0196', 'AXF0495']
core_batch      'Batch Number '           ['BNC00001', 'BNC00002', 'BNC00003']
uv_batch        'Batch Number '           ['BNU00001', 'BNU00002', 'BNU00003']
outer_batch     'Batch Number '           ['BNO00001', 'BNO00002', 'BNO00003']
core            'Sample'                  ['RAD-C-0001', 'RAD-C-0002', 'RAD-C-0003']
uv              'Sample'                  ['50:50 ESBOA:PEG', 'THB03-03', 'THB03-32']
outer           'Sample'                  ['2.25/38G/7', '4.5/14.7P/4', '2.25/38P/7']


### Observations

Two things stand out from this output:

1. **Column names themselves contain trailing whitespace** — for example
   `'Batch Number '` (with a space). This is visible in the output above.
   Padded column names are fragile: any code referencing them by string must
   include the exact whitespace, which is invisible in most editors.

2. **The join key structure is consistent across the three levels:**

| Level | Key format | Example |
|---|---|---|
| AXF run | `AXF` + 4-digit number | `AXF0020` |
| Core batch | `BNC` + 5-digit number | `BNC00001` |
| UV batch | `BNU` + 5-digit number | `BNU00015` |
| Outer batch | `BNO` + 5-digit number | `BNO00028` |
| Core formulation | `RAD-C-` + 4-digit number | `RAD-C-0001` |

These standards are good because they make it possible to distinguish key types by prefix alone. We will rely on this in the merge steps that follow.

## 3. Normalise Key Columns

We now fix the two issues identified above:
- Strip whitespace from **column names** across all tables
- Strip whitespace from **key values** in the specific columns confirmed to be affected



In [3]:
# Step 3a — We strip whitespace from column names across all tables
table_names = ['mix', 'result', 'core_batch', 'uv_batch', 'outer_batch', 'core', 'uv', 'outer']

for name in table_names:
    df = globals()[name]
    df.columns = df.columns.str.strip()
    globals()[name] = df

print("Column names stripped for:", table_names)

# Verify
print("\ncore_batch columns (first 4):", list(core_batch.columns[:4]))

Column names stripped for: ['mix', 'result', 'core_batch', 'uv_batch', 'outer_batch', 'core', 'uv', 'outer']

core_batch columns (first 4): ['Formulation', 'Batch Number', 'Batch Quantity (kg)', 'Viscosity (cP) @ RT 5 RPM']


### Step 3b — We check key values for whitespace

Column names are now clean. We now check whether the **values** inside
the key columns also carry whitespace. We check the full tables —
not a sample — so the result is conclusive.

In [4]:
# key_map uses clean column names (no trailing spaces) from this point forward
key_map = {
    'mix'         : 'batch_key',
    'result'      : 'batch_key',
    'core_batch'  : 'Batch Number',
    'uv_batch'    : 'Batch Number',
    'outer_batch' : 'Batch Number',
    'core'        : 'Sample',
    'uv'          : 'Sample',
    'outer'       : 'Sample',
}


print(f"{'Table':<25} {'Key column':<20} {'Leading':>10} {'Trailing':>10} {'Status':>10}")
print("-" * 80)

for table_name, key_col in key_map.items():
    df = globals()[table_name]
    series = df[key_col].dropna().astype(str)

    leading  = series.str.startswith(' ').sum()
    trailing = series.str.endswith(' ').sum()


    status = 'DIRTY' if (leading + trailing) > 0 else 'CLEAN'

    print(f"{table_name:<25} {key_col:<20} {leading:>10} {trailing:>10} {status:>10}")

Table                     Key column              Leading   Trailing     Status
--------------------------------------------------------------------------------
mix                       batch_key                     0          0      CLEAN
result                    batch_key                     0          0      CLEAN
core_batch                Batch Number                  0          0      CLEAN
uv_batch                  Batch Number                  0          0      CLEAN
outer_batch               Batch Number                  0          0      CLEAN
core                      Sample                        0          0      CLEAN
uv                        Sample                        0          4      DIRTY
outer                     Sample                        0          0      CLEAN



### Findings
Output reveals that only UV has trailing whitespace issue. Therefore, must be cleaned


In [5]:
def strip_col_values(df, col):
    """Strip leading/trailing whitespace from a single column's values."""
    df[col] = df[col].astype(str).str.strip()
    return df

# mix and result use batch_key — already cleaned, no stripping needed
# uv Sample still needs stripping here
uv = strip_col_values(uv, 'Sample')

# Confirm
checks = [
    (uv, 'Sample', 'uv'),
]
for df, col, label in checks:
    trailing = df[col].str.endswith(' ').sum()
    status   = 'Checked' if trailing == 0 else 'Error'
    print(f"{status}  {label}['{col}'] trailing spaces remaining: {trailing}")



Checked  uv['Sample'] trailing spaces remaining: 0


### Step 3c — Check key coverage across full tables

We verify that every batch ID used in `mix` exists in the corresponding
batch table. A missing match would produce silent NaNs in the merged dataset. That is; a production run would lose its QC record with no error raised.

In [6]:
# Step 3c — Checking key coverage across full tables

checks_coverage = [
    (mix, 'Core used',  core_batch,  'Batch Number', 'Core'),
    (mix, 'UV used',    uv_batch,    'Batch Number', 'UV'),
    (mix, 'Outer used', outer_batch, 'Batch Number', 'Outer'),
]

for mix_df, mix_col, batch_df, batch_col, label in checks_coverage:
    mix_keys   = set(mix_df[mix_col].dropna().astype(str).str.strip())
    batch_keys = set(batch_df[batch_col].dropna().astype(str).str.strip())

    in_mix_not_batch = mix_keys - batch_keys
    in_batch_not_mix = batch_keys - mix_keys

    print(f"\n── {label} ──")
    print(f"  Batch IDs in mix but NOT in {label.lower()}_batch : {len(in_mix_not_batch)}")
    print(f"  Batch IDs in {label.lower()}_batch but NOT in mix : {len(in_batch_not_mix)}")

    if in_mix_not_batch:
        print(f"  WARNING: Unresolvable keys (genuine missing records):")
        for k in sorted(in_mix_not_batch):
            print(f"    {k}")
    else:
        print(f"  SUCCESS: Every batch ID in mix has a QC record.")
        print(f"  The {len(in_batch_not_mix)} unmatched {label.lower()}_batch entries are expected:")
        print(f"  not every batch made will proceed to full production assembly.")


── Core ──
  Batch IDs in mix but NOT in core_batch : 0
  Batch IDs in core_batch but NOT in mix : 52
  SUCCESS: Every batch ID in mix has a QC record.
  The 52 unmatched core_batch entries are expected:
  not every batch made will proceed to full production assembly.

── UV ──
  Batch IDs in mix but NOT in uv_batch : 0
  Batch IDs in uv_batch but NOT in mix : 65
  SUCCESS: Every batch ID in mix has a QC record.
  The 65 unmatched uv_batch entries are expected:
  not every batch made will proceed to full production assembly.

── Outer ──
  Batch IDs in mix but NOT in outer_batch : 0
  Batch IDs in outer_batch but NOT in mix : 67
  SUCCESS: Every batch ID in mix has a QC record.
  The 67 unmatched outer_batch entries are expected:
  not every batch made will proceed to full production assembly.


## 4. Merge Strategy

With keys and values confirmed clean, we define the merge architecture
before writing any join code.

### Why NOT merge everything flat?

One core formulation (e.g. `RAD-C-0001`) maps to **many batches**, and
one batch maps to **many AXF runs**. Merging the formulation ingredient
tables directly would create:
- **Row explosion** — one row per (run × formulation ingredient combination)
- **Column redundancy** — 80+ UV ingredient columns repeated for every run using the same formula

### What we discovered about `mix`

Before designing the merge, we inspected what `mix` already carries.
It turned out to be more complete than expected:

- **Material identity** — batch IDs and formulation IDs for core, UV, and outer
- **Material properties at point of use** — viscosity for core, UV, outer, and emulsion
- **Process parameters** — flow rates, membrane, insert, cure rig, impellor, energy
- **Amounts** — core and UV masses, core loading %

This means the batch tables (`core_batch`, `uv_batch`, `outer_batch`)
would add mostly redundant information — viscosity is already recorded
in `mix` at the point of use, and the remaining batch QC fields (operator
initials, notes, QC additions) are free-text and not suitable for analysis.

### Final architecture

ANALYSIS DATASET:
mix (628 runs) **LEFT JOIN** `result` on `batch_key` - durability outcomes

Result: **770 rows** — 628 runs, +142 extra rows from runs
tested under multiple conditions (retained, not deduplicated)

**Why LEFT JOIN:** preserves all 628 production runs, including the
**151 (24%)** not yet submitted for durability testing. Dropping them
would silently remove nearly a quarter of production history.

**Why batch tables are lookup only:** `mix` already records viscosity
at point of use for all three materials. The remaining batch QC fields
are either free-text or sparsely populated. They are available for
audit and traceability but add no analytical value to the merged dataset.

**Why formulation tables are lookup only:** the `uv` table has 88
ingredient columns across 323 formulations. The formulation ID in
`axf_df` serves as a foreign key — ingredient detail is joined only
when a model question specifically requires it.

---
> McKinney, W. (2010). Data Structures for Statistical Computing in Python.
> *Proceedings of the 9th Python in Science Conference.*
> Many-to-one joins on surrogate keys are the recommended pattern when the
> reference table has a one-to-many relationship with the fact table.

##5. Renaming batch reference columns

In [7]:
# Rename batch ID columns — we give batch ID columns unambiguous names before joining
core_batch  = core_batch.rename(columns={
    'Batch Number' : 'core_batch_id',
    'Formulation'  : 'Core Formulation'
})
uv_batch    = uv_batch.rename(columns={
    'Batch Number' : 'uv_batch_id',
    'Formulation'  : 'UV formulation'
})
outer_batch = outer_batch.rename(columns={
    'Batch Number' : 'outer_batch_id',
    'Formulation'  : 'outer_formulation'
})

mix = mix.rename(columns={
    'Core used'  : 'core_batch_id',
    'UV used'    : 'uv_batch_id',
    'Outer used' : 'outer_batch_id'
})



# Confirm keys exist
print("Columns renamed")
print("  mix keys     :", [c for c in mix.columns if 'batch_id' in c])
print('result key ', 'batch_key' if 'batch_key' in result.columns else 'not found')
print("  core_batch id:", 'core_batch_id' if 'core_batch_id' in core_batch.columns else 'not found')


Columns renamed
  mix keys     : ['core_batch_id', 'uv_batch_id', 'outer_batch_id']
result key  batch_key
  core_batch id: core_batch_id


## 5. Main Merge: `mix` + `result`

Before merging, we check the distribution of `result` — how many result
rows exist per AXF run. This determines the shape of the merged dataset.


In [8]:
result_counts = result['batch_key'].value_counts()

print("Result rows per AXF run:")
print(f"  Total result rows        : {len(result)}")
print(f"  Unique batch_key     : {result['batch_key'].nunique()}")
print(f"  Runs with 1 result       : {(result_counts == 1).sum()}")
print(f"  Runs with 2+ results     : {(result_counts > 1).sum()}")
print(f"  Max results for one run  : {result_counts.max()}")
print(f"\nTop 5 runs with most results:")
print(result_counts.head().to_string())

Result rows per AXF run:
  Total result rows        : 679
  Unique batch_key     : 550
  Runs with 1 result       : 435
  Runs with 2+ results     : 115
  Max results for one run  : 5

Top 5 runs with most results:
batch_key
AXF0602    5
AXF0089    4
AXF0050    4
AXF0611    3
AXF0057    3


### What this tells us

| | |
|---|---|
| 679 result rows across 550 unique AXF runs | On average ~1.2 results per run |
| 435 runs with exactly 1 result | 79% of tested runs — the standard case |
| 115 runs with 2+ results | 21% — same batch tested under multiple conditions |
| Max 5 results for one run | `AXF0602` — likely a systematic parameter screen |

The 115 runs with multiple results are not duplicates — they represent
deliberate repeat testing under varied conditions (different coatings,
timepoints, or humidity protocols). This is scientifically valuable:
the same capsule batch observed under multiple conditions gives us more
signal for modelling.

After the left join, `axf_df` will have more rows than `mix`. The
extra rows are these multi-result observations — each one is a
legitimate data point and should be retained, not deduplicated.

One worthy observation: `AXF0602` with 5 results suggests it was used as
a reference or stress test batch. It may appear as an influential
point in modelling. That means, it is worth keeping an eye on during EDA.

In [9]:
axf_df = pd.merge(mix, result, on='batch_key', how='left', suffixes=('_mix', '_result'))


n_mix         = len(mix)
n_merged      = len(axf_df)
n_with_result = axf_df['% dry after 24h in humidity'].notna().sum()
n_no_result   = axf_df['% dry after 24h in humidity'].isna().sum()

print(f"Mix + Result merged")
print(f"   mix rows (production runs)        : {n_mix}")
print(f"   axf_df rows after merge           : {n_merged}  (+{n_merged - n_mix} from multiple results per run)")
print(f"   Rows with a durability result     : {n_with_result}")
print(f"   Rows with no result yet (NaN)     : {n_no_result}")

Mix + Result merged
   mix rows (production runs)        : 628
   axf_df rows after merge           : 770  (+142 from multiple results per run)
   Rows with a durability result     : 619
   Rows with no result yet (NaN)     : 151


After the left join:

| | |
|---|---|
| `mix` rows (production runs) | 628 |
| `axf_df` rows after merge | 770 |
| Extra rows | +142 from multi-result runs |
| Rows with a durability result | 619 (80%) |
| Rows with no result yet | 151 (20%) |

151 runs have no durability result yet — these are active production
runs that have not yet been submitted for QC testing. They are retained
in the dataset with NaN in the result columns. Dropping them would
silently remove 20% of production history from any future analysis.

In [10]:
# Inspect all columns after merge — look for duplication or unexpected suffixes
print(f"axf_df shape after merge: {axf_df.shape}")
print(f"\nAll columns ({axf_df.shape[1]}):")
for i, c in enumerate(axf_df.columns):
    print(f"  {i:>2}  {c}")

axf_df shape after merge: (770, 47)

All columns (47):
   0  AXF Number
   1  Date of experiment
   2  Aims & Hypothesis
   3  core_batch_id
   4  Core Formulation
   5  Core Viscosity (cP)
   6  Amount of Core Used (g)
   7  uv_batch_id
   8  UV formulation_mix
   9  UV viscosity (cP)
  10  Amount of UV Used (g)
  11  Emulsion viscosity (cP)
  12  Emulsion Gear Pump used
  13  Single Emulsion (SE) Number
  14  Vessel Used
  15  Impellor Used
  16  Mass of emulsion used / g
  17  Impellor Speed (rpm)
  18  Core Loading (%)
  19  outer_batch_id
  20  Outer viscosity (cP)
  21  Outer Gear Pump Used
  22  Dispersed Flow Rate (mL/min)
  23  Continuous Flow Rate (mL/min)
  24  Flow Rate Ratio
  25  Cure Rig Used
  26  Membrane Used (µm)
  27  Insert used (mm)
  28  Length of curing tubing (m)
  29  Temperature of Outer Phase at end of experiment / C
  30  UV Power (J s-1)
  31  Curing Energy (kJ g-1)
  32  batch_key
  33  Batch number
  34  UV formulation_result
  35  Core formulation
  36 

### Observations — column inspection after merge

Two issues are visible:

1. **`UV formulation_mix` / `UV formulation_result`** — suffixed correctly by pandas
   because both `mix` and `result` had the exact same column name `'UV formulation'`

2. **`Core Formulation` / `Core formulation`** — pandas did NOT suffix these because
   it treated them as different columns — `mix` had capital F `'Core Formulation'`,
   `result` had lowercase f `'Core formulation'`. Both survived the merge unchanged.

Both the result versions carry identical information to the mix versions.
We drop the result versions and rename the `_mix` suffix in the cell below.

In [11]:
# Drop result versions — identical content to mix versions
axf_df = axf_df.drop(columns=[
    'UV formulation_result',
    'Core formulation'
])

# Rename _mix suffix now clean
axf_df = axf_df.rename(columns={
    'UV formulation_mix' : 'UV formulation'
})

print(f"Shape after cleanup : {axf_df.shape}")
print(f"\nFormulation columns remaining:")
for c in axf_df.columns:
    if 'formulation' in c.lower():
        print(f"  {c}")

Shape after cleanup : (770, 45)

Formulation columns remaining:
  Core Formulation
  UV formulation


## 6. Final Dataset Summary

In [12]:
print("=" * 60)
print("ANALYSIS DATASET -  axf_df")
print(f"  Rows                          : {axf_df.shape[0]}")
print(f"  Columns                       : {axf_df.shape[1]}")
print(f"  Rows with durability result   : {axf_df['% dry after 24h in humidity'].notna().sum()}")
print(f"  Rows without result yet (NaN) : {axf_df['% dry after 24h in humidity'].isna().sum()}")
print(f"  Unique AXF runs               : {axf_df['batch_key'].nunique()}")
print(f"  Unique core formulations      : {axf_df['Core Formulation'].nunique()}")
print(f"  Unique UV formulations        : {axf_df['UV formulation'].nunique()}")
print("=" * 60)

print("\nLOOKUP TABLES  (retained separately, join on demand):")
print(f"  core        : {core.shape[0]} formulations × {core.shape[1]-1} ingredient columns")
print(f"  uv          : {uv.shape[0]} formulations × {uv.shape[1]-1} ingredient columns")
print(f"  outer       : {outer.shape[0]} formulations × {outer.shape[1]-1} ingredient columns")
print(f"  core_batch  : {core_batch.shape[0]} batches × {core_batch.shape[1]-1} QC columns")
print(f"  uv_batch    : {uv_batch.shape[0]} batches × {uv_batch.shape[1]-1} QC columns")
print(f"  outer_batch : {outer_batch.shape[0]} batches × {outer_batch.shape[1]-1} QC columns")

print("\nAll formulation FK columns in axf_df:")
for c in axf_df.columns:
    if any(x in c for x in ['Formulation', 'formulation', 'batch_id']):
        print(f"  {c}")

ANALYSIS DATASET -  axf_df
  Rows                          : 770
  Columns                       : 45
  Rows with durability result   : 619
  Rows without result yet (NaN) : 151
  Unique AXF runs               : 612
  Unique core formulations      : 15
  Unique UV formulations        : 42

LOOKUP TABLES  (retained separately, join on demand):
  core        : 54 formulations × 22 ingredient columns
  uv          : 323 formulations × 88 ingredient columns
  outer       : 7 formulations × 7 ingredient columns
  core_batch  : 117 batches × 11 QC columns
  uv_batch    : 177 batches × 8 QC columns
  outer_batch : 255 batches × 8 QC columns

All formulation FK columns in axf_df:
  core_batch_id
  Core Formulation
  uv_batch_id
  UV formulation
  outer_batch_id


## Summary

| Output | Rows | Columns | Description |
|---|---|---|---|
| `axf_df` | 770 | 45 | Analysis dataset — all process inputs + durability outcomes |
| `core` | 54 | 23 | Core ingredient recipes |
| `uv` | 323 | 89 | UV ingredient recipes |
| `outer` | 7 | 8 | Outer ingredient recipes |
| `core_batch` | 117 | 12 | Core batch QC records |
| `uv_batch` | 177 | 9 | UV batch QC records |
| `outer_batch` | 255 | 9 | Outer batch QC records |

The 770 rows in `axf_df` represent 628 unique AXF production runs.
The extra 142 rows are runs tested under multiple durability conditions —
retained as separate observations for modelling.

151 rows have no durability result yet. These are active production
runs not yet submitted for QC testing and are preserved with NaN.

Next step: `albotherm_03_eda.ipynb`

## 8. Save Outputs

In [ ]:
axf_df.to_csv('../data/processed/axf_merged.csv', index=False)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>